라벨은 y = (category == '상장폐지')  
("정상기업 공시" 데이터까지 붙이면, 그쪽을 0으로 넣고 이쪽을 1로 두면 됨.)

In [1]:
!pip install kiwipiepy -q
!pip install -q konlpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 8.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.9/495.9 kB 22.1 MB/s eta 0:00:00


## 0) 공통: 데이터 로드 + 텍스트 결합 + 라벨 생성

In [2]:
import pandas as pd
import numpy as np
from collections import Counter
from kiwipiepy import Kiwi
from tqdm import tqdm

# 1. 데이터 로드 및 병합 (year 칼럼 포함)
# 각 파일에 'corp_code', 'year', 'y', 'text'가 있다고 가정합니다.
df_audit = pd.read_csv("dataset_AUDIT.csv", encoding='utf-8-sig')
df_mda = pd.read_csv("dataset_MDA.csv", encoding='utf-8-sig')
df_etc = pd.read_csv("dataset_ETC.csv", encoding='utf-8-sig')

# 섹션별 텍스트 구분을 위해 컬럼명 변경 후 병합
df_audit = df_audit.rename(columns={'text': 'text_audit'})
df_mda = df_mda.rename(columns={'text': 'text_mda'})
df_etc = df_etc.rename(columns={'text': 'text_etc'})

# corp_code, year, y를 기준으로 중복 없이 병합
df = pd.merge(df_audit, df_mda, on=['corp_code', 'target_year', 'y'], how='inner')
df = pd.merge(df, df_etc, on=['corp_code', 'target_year', 'y'], how='inner')

# 세 섹션의 텍스트를 하나로 결합 (감성사전용)
df['text'] = df['text_audit'].fillna('') + " " + df['text_mda'].fillna('') + " " + df['text_etc'].fillna('')

print("="*60)
print(f"데이터 병합 완료 (year 칼럼 유지)")
print(f"총 샘플: {len(df):,}개")
print(f"컬럼: {list(df.columns)}")
print("="*60)

데이터 병합 완료 (year 칼럼 유지)
총 샘플: 28,222개
컬럼: ['corp_code', 'target_year', 'y', 'text_audit', 'text_mda', 'text_etc', 'text']


## 1) 방식 1: 말뭉치 기반 감성사전 구축 + 문서 감성 피처

1-1) 설치 + 명사 추출기

1-2) 감성사전 구축(문서빈도 기반) + 중립/희소 단어 제거

In [5]:
from collections import Counter, defaultdict
import re, math
from kiwipiepy import Kiwi

kiwi = Kiwi()

# (권장) 고유명사 제거
NOUN_TAGS = {"NNG","NP","NNB","NR"}

MAX_CHARS = 5000
df["text_cut"] = df["text"].astype(str).str.slice(0, MAX_CHARS)

def extract_nouns_fast(text):
    nouns = []
    for tok in kiwi.tokenize(text):
        if tok.tag in NOUN_TAGS:
            w = tok.form
            if len(w) >= 2:
                nouns.append(w)
    return nouns

# ✅ 여기서 nouns를 반드시 만든다
df["nouns"] = df["text_cut"].map(extract_nouns_fast)

# -----------------------------
# DF(문서빈도) + Firm-DF(기업빈도)
# -----------------------------
df_count_all = Counter()
df_count_0 = Counter()
df_count_1 = Counter()

firm_set_0 = defaultdict(set)
firm_set_1 = defaultdict(set)

hangul_re = re.compile(r"^[가-힣]{2,}$")

for nouns, y, corp in zip(df["nouns"], df["y"], df["corp_code"]):
    uniq = set(nouns)
    uniq = {t for t in uniq if hangul_re.match(t)}

    df_count_all.update(uniq)
    if y == 0:
        df_count_0.update(uniq)
        for t in uniq:
            firm_set_0[t].add(corp)
    else:
        df_count_1.update(uniq)
        for t in uniq:
            firm_set_1[t].add(corp)

N0 = int((df["y"] == 0).sum())
N1 = int((df["y"] == 1).sum())

# -----------------------------
# 파라미터
# -----------------------------
min_df_all      = 20
neutral_th_z    = 2.0

min_df_each     = 5

min_df_excl     = 30
min_firm_excl   = 10
max_df_other    = 2

def z_score(n0, n1, N0, N1, eps=1e-12):
    p0 = n0 / max(N0, 1)
    p1 = n1 / max(N1, 1)
    p  = (n0 + n1) / max(N0 + N1, 1)
    se = math.sqrt(max(p * (1 - p) * (1/max(N0,1) + 1/max(N1,1)), eps))
    return (p0 - p1) / se

lexicon = {}

for term, n_all in df_count_all.items():
    if n_all < min_df_all:
        continue

    n0 = df_count_0.get(term, 0)
    n1 = df_count_1.get(term, 0)

    shared_ok = (n0 >= min_df_each) and (n1 >= min_df_each)

    firm0 = len(firm_set_0.get(term, set()))
    firm1 = len(firm_set_1.get(term, set()))

    exclusive_0_ok = (n0 >= min_df_excl) and (n1 <= max_df_other) and (firm0 >= min_firm_excl)
    exclusive_1_ok = (n1 >= min_df_excl) and (n0 <= max_df_other) and (firm1 >= min_firm_excl)

    if not (shared_ok or exclusive_0_ok or exclusive_1_ok):
        continue

    z = z_score(n0, n1, N0, N1)

    if abs(z) < neutral_th_z:
        continue

    lexicon[term] = z

print("lexicon size =", len(lexicon))
top_pos = sorted(lexicon.items(), key=lambda x: x[1], reverse=True)[:20]
top_neg = sorted(lexicon.items(), key=lambda x: x[1])[:20]
print("Top + terms (normal-heavy):", top_pos)
print("Top - terms (default-heavy):", top_neg)


lexicon size = 2262
Top + terms (normal-heavy): [('순이익', 22.682958379322276), ('증가', 20.96852530153319), ('이익', 20.381577641070688), ('대비', 19.455972255237533), ('정보', 19.203344457173067), ('전년', 19.09956041186191), ('성장', 18.769259322382595), ('환경', 17.54205628129702), ('비율', 17.296215040966473), ('예측', 17.225767112800337), ('미래', 17.21386342595962), ('요인', 15.861229683694324), ('예상', 15.570711017735938), ('실적', 15.412483230226833), ('제품', 15.39506294389001), ('작성', 15.32849899291449), ('다양', 15.13117423163003), ('글로벌', 15.103250025859836), ('안정', 15.037215957252812), ('지속', 14.800846290213238)]
Top - terms (default-heavy): [('거절', -43.74834287840704), ('기말', -30.9619066572035), ('폐지', -29.629085439659907), ('회생', -28.19663451005845), ('의문', -24.069499146137133), ('내역', -22.46369789516945), ('정지', -21.61250297588865), ('인가', -21.238251760900894), ('좌우', -20.409874065568747), ('순손실', -20.30195915077684), ('계획안', -19.7480453792667), ('성패', -19.716291076764733), ('제기', -19.60680616323890

In [8]:
top_pos

[('순이익', 22.682958379322276),
 ('증가', 20.96852530153319),
 ('이익', 20.381577641070688),
 ('대비', 19.455972255237533),
 ('정보', 19.203344457173067),
 ('전년', 19.09956041186191),
 ('성장', 18.769259322382595),
 ('환경', 17.54205628129702),
 ('비율', 17.296215040966473),
 ('예측', 17.225767112800337),
 ('미래', 17.21386342595962),
 ('요인', 15.861229683694324),
 ('예상', 15.570711017735938),
 ('실적', 15.412483230226833),
 ('제품', 15.39506294389001),
 ('작성', 15.32849899291449),
 ('다양', 15.13117423163003),
 ('글로벌', 15.103250025859836),
 ('안정', 15.037215957252812),
 ('지속', 14.800846290213238)]

In [9]:
top_neg

[('거절', -43.74834287840704),
 ('기말', -30.9619066572035),
 ('폐지', -29.629085439659907),
 ('회생', -28.19663451005845),
 ('의문', -24.069499146137133),
 ('내역', -22.46369789516945),
 ('정지', -21.61250297588865),
 ('인가', -21.238251760900894),
 ('좌우', -20.409874065568747),
 ('순손실', -20.30195915077684),
 ('계획안', -19.7480453792667),
 ('성패', -19.716291076764733),
 ('제기', -19.606806163238904),
 ('법인', -19.537376851206748),
 ('심의', -19.42659814615363),
 ('주석', -19.213877943263512),
 ('적격', -18.691280245444716),
 ('횡령', -18.046758031481275),
 ('절차', -18.005568389131124),
 ('법원', -17.779284712085218)]

1-3) 문서별 감성 피처 생성 (XGBoost 입력용)

권장 피처(설명가능성 유지):

lex_sent_mean: 문서 내 (score의 TF가중 평균)

lex_sent_sum: score*tf 합

lex_pos_tf, lex_neg_tf: 긍/부정 단어 출현 빈도(토큰수)

lex_pos_cnt, lex_neg_cnt: 긍/부정 단어 "종류 수"(유니크)

lex_abs_mean: |score| 평균(강도)

In [6]:
def make_lex_features(nouns):
    if not nouns:
        return {
            "lex_sent_mean": 0.0,
            "lex_sent_sum": 0.0,
            "lex_pos_tf": 0.0,
            "lex_neg_tf": 0.0,
            "lex_pos_cnt": 0.0,
            "lex_neg_cnt": 0.0,
            "lex_abs_mean": 0.0,
            "lex_covered_tf": 0.0,   # 사전에 잡힌 토큰 수
        }
    tf = Counter(nouns)
    sent_sum = 0.0
    covered_tf = 0

    pos_tf = 0
    neg_tf = 0
    pos_terms = set()
    neg_terms = set()
    abs_scores = []

    for term, f in tf.items():
        if term not in lexicon:
            continue
        s = lexicon[term]
        sent_sum += s * f
        covered_tf += f
        abs_scores.extend([abs(s)] * f)
        if s > 0:
            pos_tf += f
            pos_terms.add(term)
        else:
            neg_tf += f
            neg_terms.add(term)

    if covered_tf == 0:
        return {
            "lex_sent_mean": 0.0,
            "lex_sent_sum": 0.0,
            "lex_pos_tf": 0.0,
            "lex_neg_tf": 0.0,
            "lex_pos_cnt": 0.0,
            "lex_neg_cnt": 0.0,
            "lex_abs_mean": 0.0,
            "lex_covered_tf": 0.0,
        }

    return {
        "lex_sent_mean": sent_sum / covered_tf,
        "lex_sent_sum": sent_sum,
        "lex_pos_tf": float(pos_tf),
        "lex_neg_tf": float(neg_tf),
        "lex_pos_cnt": float(len(pos_terms)),
        "lex_neg_cnt": float(len(neg_terms)),
        "lex_abs_mean": float(np.mean(abs_scores)) if abs_scores else 0.0,
        "lex_covered_tf": float(covered_tf),
    }

lex_feat = df["nouns"].apply(make_lex_features).apply(pd.Series)
df_lex = pd.concat([df, lex_feat], axis=1)

df_lex[["lex_sent_mean","lex_pos_tf","lex_neg_tf","lex_pos_cnt","lex_neg_cnt","lex_abs_mean","lex_covered_tf"]].head()

,lex_sent_mean,lex_pos_tf,lex_neg_tf,lex_pos_cnt,lex_neg_cnt,lex_abs_mean,lex_covered_tf
0,-0.243965,347.0,190.0,71.0,22.0,8.957345,537.0
1,-0.516110,288.0,167.0,68.0,18.0,9.051981,455.0
2,0.288011,285.0,150.0,72.0,16.0,9.120964,435.0
3,-0.119294,282.0,161.0,72.0,18.0,9.178569,443.0
4,0.022073,270.0,152.0,68.0,20.0,9.073849,422.0


In [7]:
# 3. 결과 저장 (year 칼럼 반드시 포함)
lex_feature_cols = [
    "lex_sent_mean", "lex_sent_sum", "lex_pos_tf", "lex_neg_tf",
    "lex_pos_cnt", "lex_neg_cnt", "lex_abs_mean", "lex_covered_tf"
]

# 출력을 위해 필요한 컬럼만 추출 (year 포함)
out_lex = df_lex[["corp_code", "target_year", "y"] + lex_feature_cols].copy()
out_lex.to_csv("text_features_lexicon_with_year.csv", index=False, encoding="utf-8-sig")

print(f"저장 완료: text_features_lexicon_with_year.csv")
print(out_lex.head())

저장 완료: text_features_lexicon_with_year.csv
   corp_code  target_year  y  lex_sent_mean  lex_sent_sum  lex_pos_tf  \
0     126380         2024  0      -0.243965   -131.009224       347.0   
1     126380         2023  0      -0.516110   -234.829848       288.0   
2     126380         2022  0       0.288011    125.284673       285.0   
3     126380         2021  0      -0.119294    -52.847127       282.0   
4     126380         2020  0       0.022073      9.314605       270.0   

   lex_neg_tf  lex_pos_cnt  lex_neg_cnt  lex_abs_mean  lex_covered_tf  
0       190.0         71.0         22.0      8.957345           537.0  
1       167.0         68.0         18.0      9.051981           455.0  
2       150.0         72.0         16.0      9.120964           435.0  
3       161.0         72.0         18.0      9.178569           443.0  
4       152.0         68.0         20.0      9.073849           422.0  
